In [14]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

# If you already set these, this cell will keep them.
os.environ.setdefault("DASHSCOPE_API_KEY", os.getenv("DASHSCOPE_API_KEY", ""))
os.environ.setdefault(
    "DASHSCOPE_API_BASE",
    os.getenv("DASHSCOPE_API_BASE", "https://dashscope-us.aliyuncs.com/compatible-mode/v1"),
)

# LLLM (LiteLLM invoker) reads OpenAI-style env vars for compatible endpoints.
os.environ["OPENAI_API_KEY"] = os.environ["DASHSCOPE_API_KEY"]
os.environ["OPENAI_API_BASE"] = os.environ["DASHSCOPE_API_BASE"]
os.environ["OPENAI_BASE_URL"] = os.environ["DASHSCOPE_API_BASE"]


def find_repo_root(start: Path) -> Path:
    """Find repo root that contains lllm/__init__.py."""
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "lllm" / "__init__.py").exists():
            return p
    raise RuntimeError("Could not locate repo root containing lllm/__init__.py")


repo_root = find_repo_root(Path.cwd())
repo_parent = repo_root.parent.resolve()

# Remove wrong higher-level entries that trigger namespace import.
cleaned = []
for p in sys.path:
    try:
        if Path(p).resolve() == repo_parent:
            continue
    except Exception:
        pass
    cleaned.append(p)
sys.path = cleaned

# Ensure correct repo root is first in import path.
repo_root_str = str(repo_root)
if repo_root_str in sys.path:
    sys.path.remove(repo_root_str)
sys.path.insert(0, repo_root_str)

# If a wrong namespace module was imported earlier, clear it.
for mod in list(sys.modules):
    if mod == "lllm" or mod.startswith("lllm."):
        del sys.modules[mod]


def ensure_dep(import_name: str, pip_spec: str) -> None:
    """Install dependency into current kernel env if missing."""
    try:
        importlib.import_module(import_name)
    except Exception:
        print(f"Installing missing dependency: {pip_spec}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_spec])


# Required by this repo's lllm package.
ensure_dep("pydantic", "pydantic>=2")
ensure_dep("yaml", "pyyaml")
ensure_dep("litellm", "litellm")

print("Python executable:", sys.executable)
print("repo_root:", repo_root)
print("sys.path[0]:", sys.path[0])
print("OPENAI_API_BASE:", os.environ["OPENAI_API_BASE"])
print("OPENAI_API_KEY set:", bool(os.environ["OPENAI_API_KEY"]))

Python executable: /Users/f006ncc/Documents/phd research/LLM/lllm/Packages/.venv/bin/python
repo_root: /Users/f006ncc/Documents/phd research/LLM/lllm
sys.path[0]: /Users/f006ncc/Documents/phd research/LLM/lllm
OPENAI_API_BASE: https://dashscope-us.aliyuncs.com/compatible-mode/v1
OPENAI_API_KEY set: True


In [9]:
import lllm, sys; print(lllm, getattr(lllm, "__file__", None)); print(sys.path[:3])

<module 'lllm' from '/Users/f006ncc/Documents/phd research/LLM/lllm/lllm/__init__.py'> /Users/f006ncc/Documents/phd research/LLM/lllm/lllm/__init__.py
['/Users/f006ncc/Documents/phd research/LLM/lllm', '/opt/homebrew/Cellar/python@3.13/3.13.2/Frameworks/Python.framework/Versions/3.13/lib/python313.zip', '/opt/homebrew/Cellar/python@3.13/3.13.2/Frameworks/Python.framework/Versions/3.13/lib/python3.13']


In [15]:
from lllm.core.runtime import load_runtime
from lllm.core.config import resolve_config
from lllm.core.tactic import build_tactic
from lllm.invokers.litellm import LiteLLMInvoker

# Hotfix for providers (e.g., Qwen via DashScope) that return response_cost=None.
_original_build_usage = LiteLLMInvoker._build_usage


def _build_usage_safe(self, usage_dict, response_obj, model):
    usage_dict = usage_dict or {}
    usage = _original_build_usage(self, usage_dict, response_obj, model)

    # Ensure all numeric cost fields are floats, never None.
    for key in (
        "response_cost",
        "prompt_cost",
        "completion_cost",
        "input_cost_per_token",
        "output_cost_per_token",
        "cache_read_input_token_cost",
    ):
        val = usage.get(key)
        usage[key] = float(val) if val is not None else 0.0
    return usage


LiteLLMInvoker._build_usage = _build_usage_safe

pkg_toml = repo_root / "Packages" / "time_series_analytics" / "lllm.toml"
runtime = load_runtime(
    toml_path=str(pkg_toml),
    name="ts_notebook_runtime",
    discover_shared_packages=False,
)

# Use a Qwen model through DashScope-compatible endpoint.
cfg = resolve_config("time_series_analytics:default", runtime=runtime)
cfg.setdefault("global", {})["model_name"] = "openai/qwen-plus"

tactic = build_tactic(cfg, runtime=runtime)
print("Tactic ready:", tactic.name)

Tactic ready: time_series_analysis


In [16]:
import csv
from dataclasses import dataclass

from tactics.time_series_analysis import TimeSeriesTask


def read_csv_as_text(path: Path, max_rows: int = 300) -> str:
    with path.open("r", encoding="utf-8", newline="") as fh:
        reader = csv.DictReader(fh)
        if not reader.fieldnames:
            raise ValueError(f"CSV has no header: {path}")

        rows = []
        for i, row in enumerate(reader):
            rows.append(row)
            if i + 1 >= max_rows:
                break

    header = ",".join(reader.fieldnames)
    body = [",".join(str(r.get(col, "")) for col in reader.fieldnames) for r in rows]
    return "\n".join([header] + body)


@dataclass
class NotebookTimeSeriesAgent:
    tactic: object

    def run(
        self,
        csv_path: str | Path,
        timestamp_col: str = "date",
        value_col: str = "sales",
        horizon: int = 7,
        frequency: str = "D",
        objective: str = "Detect anomalies and forecast future values.",
    ) -> dict:
        series_data = read_csv_as_text(Path(csv_path))
        task = TimeSeriesTask(
            series_data=series_data,
            timestamp_col=timestamp_col,
            value_col=value_col,
            horizon=horizon,
            frequency=frequency,
            objective=objective,
        )
        result = self.tactic(task)
        return result.model_dump()


agent = NotebookTimeSeriesAgent(tactic=tactic)
print("Agent initialized.")

Agent initialized.


In [17]:
# Demo run with the package's sample CSV.
demo_csv = repo_root / "Packages" / "time_series_analytics" / "demo_sales.csv"

output = agent.run(
    demo_csv,
    timestamp_col="date",
    value_col="sales",
    horizon=7,
    frequency="D",
    objective="Detect anomalies and forecast next-week sales demand.",
)

output

/var/folders/62/ff74mwld5_v26kltzrp79qzw0000gq/T/ipykernel_83778/3331148048.py:46: UserWarning: No LogStore configured for tactic 'time_series_analysis'. Session data will not be persisted. Pass a LogStore instance via the log_store parameter.
  result = self.tactic(task)


{'summary': 'Anomaly detected on 2026-05-05 (sales = 180), likely a one-off event. Forecast assumes a mild upward trend (+1.4/day) from a baseline of 125, with flat weekly seasonality due to insufficient data. All forecasts carry high uncertainty; 90% prediction intervals reflect empirical residual spread and trend error. Critical data gaps prevent reliable modeling — immediate data collection and domain validation are required before operational use.',
 'key_patterns': ['Mild positive trend post-outlier (May 06–10): +1.4 units/day, though statistically marginal (p ≈ 0.12, n=5).',
  'No evidence of weekly or monthly seasonality — only one weekend observed; insufficient for day-of-week modeling.',
  'Baseline sales level (median of non-outlier values) is stable at 125, with low short-term volatility (std ≈ 3.8).'],
 'data_quality_issues': [{'issue': 'Severely limited sample size',
   'severity': 'critical',
   'evidence': 'Only 10 consecutive daily observations (2026-05-01 to 2026-05-10

In [ ]:
# Replace this with your own file path and columns.
my_csv = demo_csv  # e.g. Path("/absolute/path/to/your_series.csv")

my_output = agent.run(
    my_csv,
    timestamp_col="date",      # change to your timestamp column
    value_col="sales",         # change to your metric column
    horizon=14,
    frequency="D",
    objective="Create anomaly report and 14-step forecast for operations planning.",
)

my_output